# Visual-Driven Chapter Segmentation
## Sliding Window + Cosine Discontinuity Detection on Visual Features
### Video and AI: NLP and Deep Learning Applications

---

## Design Principle

The **main pipeline** (`Feature_Extraction_Optimized.ipynb`) segments the transcript using a
fixed word-count tumbling window (W = 300 words) and then *aligns* visual features to those
text-derived boundaries.  The visual signal plays no role in deciding *where* chapters begin
and end.

This notebook **inverts that priority**: visual features are extracted first, and a sliding
window + cosine-distance detector applied **directly to the ResNet-18 feature vectors**
determines all chapter boundaries.  The transcript is then sliced at those boundaries
retrospectively.  Word-count thresholding plays no role in segmentation.

```
Main pipeline (text-first)            This notebook (visual-first)
──────────────────────────────         ──────────────────────────────────────────
Stage 1  Whisper ASR → segments        Stage 1  Whisper ASR → segments
Stage 2  Word-count tumbling window    Stage 2  ResNet-18 frame extraction
Stage 3  ResNet-18 per chapter         Stage 3  Sliding window on feature vectors
Stage 4  SBERT per chapter                        → cosine distance curve
Stage 5  LLM label                                → peak detection → breakpoints
Stage 6  Flask                         Stage 4  Slice transcript at breakpoints
                                       Stage 5  SBERT per chapter
                                       Stage 6  LLM label
                                       Stage 7  Flask (unchanged)
```

### Why this works better

Lecture content changes are usually signalled visually first: a new slide appears, the
lecturer moves to the board, or a code editor opens — all of which produce an abrupt change
in the ResNet-18 feature space before the associated speech begins.  Cosine-distance peaks
in the **sliding-window feature sequence** precisely localise these transitions.  Cutting the
transcript at those positions ensures each chapter corresponds to one coherent visual segment
rather than an arbitrary 300-word block.

---

## 0. Install Dependencies

In [ ]:
%pip install -q openai-whisper torch torchvision sentence-transformers
%pip install -q opencv-python-headless transformers accelerate scipy matplotlib tqdm pillow


## 1. Imports

In [ ]:
import os, json, re, time, warnings
import numpy as np
import cv2
import torch
import whisper
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
from PIL import Image as PILImage
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')


## 2. Configuration

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
VIDEO_PATH         = 'videos/Week 01 - Embedded S.mp4'
AUDIO_PATH         = 'Output/audio.wav'
TRANSCRIPT_PATH    = 'Output/transcript_segments.json'   # Stage-1 cache
FEATURES_PATH      = 'Output/sw_frame_features.npz'      # Stage-2 cache
OUTPUT_DIR         = 'Output/sw_visual'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Stage 2: frame sampling ──────────────────────────────────────────────────
# Sample densely so the sliding window has many data points per minute.
FRAME_SAMPLE_RATE  = 15          # seconds between sampled frames

# ── Stage 3: sliding window parameters ──────────────────────────────────────
#
#   W_VIS  = window width in seconds.  Each window's feature is the mean-pool
#             of all frames whose timestamps fall inside it.
#   S_VIS  = step size.  S_VIS < W_VIS produces OVERLAPPING windows.
#
#   Overlap fraction = (W_VIS - S_VIS) / W_VIS
#
#   With W=60, S=5 → 91.7 % overlap: every 5-second slice is represented in
#   12 consecutive windows, making the cosine distance curve very smooth and
#   responsive to sustained visual changes rather than single-frame noise.
#
W_VIS              = 60          # window width  (seconds)
S_VIS              = 5           # window step   (seconds)  — must be < W_VIS

# ── Stage 3: cosine-distance peak detection ──────────────────────────────────
SMOOTH_SIGMA       = 2.0         # Gaussian σ applied to distance curve before peak-finding
PEAK_PROMINENCE    = 0.05        # minimum prominence for a peak to count as a boundary
PEAK_MIN_DIST_S    = 60          # minimum seconds between two detected boundaries

# ── Stage 4: transcript slicing ──────────────────────────────────────────────
MIN_CHAPTER_WORDS  = 60          # merge too-short chapters with the next one

# ── Models ───────────────────────────────────────────────────────────────────
WHISPER_MODEL      = 'large-v3-turbo'
SBERT_MODEL        = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL          = 'Qwen/Qwen2.5-1.5B-Instruct'
LLM_TEMPERATURE    = 0.1
LLM_TOP_P          = 0.9
LLM_MAX_TOKENS     = 256
MAX_RETRIES        = 3

print('Configuration ready')
print(f'  Frame sample rate : every {FRAME_SAMPLE_RATE}s')
print(f'  Sliding window    : W={W_VIS}s, S={S_VIS}s  '
      f'(overlap = {100*(W_VIS-S_VIS)/W_VIS:.0f}%)')


## 3. Utility Functions

In [ ]:
def fmt_time(seconds):
    seconds = max(0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def cosine_sim(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if (na > 0 and nb > 0) else 0.0


FILLER_WORDS = [
    ' um ', ' uh ', ' like ', ' you know ', ' so ', ' actually ',
    ' basically ', ' I mean ', ' yeah ', ' cool '
]

def clean_text(text):
    for fw in FILLER_WORDS:
        text = text.replace(fw, ' ')
    return re.sub(r' +', ' ', text).strip()


LOW_VALUE_KEYWORDS = [
    'introduction','intro','summary','conclusion','setup','housekeeping',
    'announcements','admin','recap','review','demo','q&a','lab',
    'policies','course overview','tutorial','basics','overview',
    'assignment','project','group formation'
]

def is_low_value(title, desc):
    text = (title + ' ' + desc).lower()
    return any(k in text for k in LOW_VALUE_KEYWORDS)


def extract_json_brace(text):
    """Brace-counting JSON extractor (identical to main pipeline)."""
    count, start = 0, None
    for i, ch in enumerate(text):
        if ch == '{':
            if start is None: start = i
            count += 1
        elif ch == '}':
            count -= 1
            if count == 0 and start is not None:
                return text[start:i+1]
    return None


print('Utilities defined')


## Stage 1 — Whisper ASR

Transcribes the lecture audio and caches the output as `transcript_segments.json`.
If the file already exists (from a previous run of the main pipeline) it is loaded directly.

In [ ]:
if os.path.exists(TRANSCRIPT_PATH):
    with open(TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
        segments = json.load(f)
    print(f'Loaded cached transcript: {len(segments)} segments')
else:
    print('Running Whisper ASR (~50 min on CPU)...')
    if not os.path.exists(AUDIO_PATH):
        os.makedirs(os.path.dirname(AUDIO_PATH) or '.', exist_ok=True)
        os.system(f'ffmpeg -i "{VIDEO_PATH}" -vn -acodec pcm_s16le -ar 16000 -ac 1 -y "{AUDIO_PATH}"')
    asr = whisper.load_model(WHISPER_MODEL)
    res = asr.transcribe(AUDIO_PATH, language='en', word_timestamps=True)
    del asr
    segments = []
    for seg in res['segments']:
        t = clean_text(seg['text'].strip())
        if t:
            segments.append({
                'id': seg['id'],
                'start_time': float(seg['start']),
                'end_time':   float(seg['end']),
                'text': t,
            })
    with open(TRANSCRIPT_PATH, 'w', encoding='utf-8') as f:
        json.dump(segments, f, indent=2)
    print(f'Transcript saved: {len(segments)} segments')

video_end_sec = float(segments[-1]['end_time'])
print(f'  Duration : {fmt_time(video_end_sec)}')
print(f'  Sample   : [{fmt_time(segments[0]["start_time"])}] {segments[0]["text"][:80]}')


## Stage 2 — ResNet-18 Dense Frame Extraction

Each frame sampled every `FRAME_SAMPLE_RATE` seconds is passed through **ResNet-18**
(ImageNet pre-trained, classification head removed) to produce a **512-dimensional feature
vector**.  The result is a matrix

$$F \in \mathbb{R}^{N \times 512}$$

where $N$ is the number of sampled frames, and a parallel timestamp array
$T \in \mathbb{R}^{N}$.

At `FRAME_SAMPLE_RATE = 15s` a 90-minute lecture produces $N \approx 360$ frames.
The features are cached to `FEATURES_PATH` so this step only runs once.

In [ ]:
# ── ResNet-18 in feature-extractor mode ─────────────────────────────────────
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = torch.nn.Identity()   # remove classifier → 512-dim output
resnet.eval()

PREPROCESS = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def frame_to_feature(bgr):
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    t    = PREPROCESS(PILImage.fromarray(rgb)).unsqueeze(0)
    return resnet(t).squeeze().numpy()   # (512,)

print('ResNet-18 ready (CPU)')


In [ ]:
if os.path.exists(FEATURES_PATH):
    data = np.load(FEATURES_PATH)
    frame_ts   = data['timestamps']   # (N,)
    frame_feat = data['features']     # (N, 512)
    print(f'Loaded cached features: {frame_feat.shape}')
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    assert cap.isOpened(), f'Cannot open video: {VIDEO_PATH}'

    fps        = cap.get(cv2.CAP_PROP_FPS)
    n_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step       = int(fps * FRAME_SAMPLE_RATE)

    ts_list, feat_list = [], []
    frame_idx = 0

    with tqdm(total=n_frames // step, desc='ResNet-18 extraction') as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % step == 0:
                ts_list.append(frame_idx / fps)
                feat_list.append(frame_to_feature(frame))
                pbar.update(1)
            frame_idx += 1
    cap.release()

    frame_ts   = np.array(ts_list,  dtype=np.float32)
    frame_feat = np.stack(feat_list).astype(np.float32)
    np.savez(FEATURES_PATH, timestamps=frame_ts, features=frame_feat)
    print(f'Features saved: {frame_feat.shape} → {FEATURES_PATH}')

print(f'\n✓  Frames extracted: N = {len(frame_ts)}')
print(f'   Timeline : {fmt_time(frame_ts[0])} → {fmt_time(frame_ts[-1])}')
print(f'   Feature  : shape = {frame_feat.shape},  dtype = {frame_feat.dtype}')


## Stage 3a — Build Sliding Windows Over Visual Feature Vectors

Immediately after feature extraction we apply a **sliding window** directly to $F$.

For each window starting at $t_k = k \cdot S_{\text{vis}}$ we collect all frames
whose timestamps lie in $[t_k,\; t_k + W_{\text{vis}})$ and compute their
**mean-pooled feature vector**:

$$\mathbf{w}_k = \frac{1}{|\mathcal{F}_k|} \sum_{i \in \mathcal{F}_k} \mathbf{f}_i
\qquad \mathcal{F}_k = \{\, i : t_k \le T_i < t_k + W_{\text{vis}} \,\}$$

This produces a sequence of window descriptors $\{\mathbf{w}_k\}$ sampled every
$S_{\text{vis}}$ seconds, each representing the *average visual state* of a
$W_{\text{vis}}$-second region.  The high overlap
$(W_{\text{vis}} - S_{\text{vis}}) / W_{\text{vis}}$
ensures the descriptor changes smoothly unless a genuine visual transition occurs.

In [ ]:
win_times   = []   # centre-of-window timestamps (seconds)
win_feats   = []   # mean-pooled 512-dim feature per window

t_start = 0.0
while t_start + W_VIS <= float(frame_ts[-1]):
    t_end = t_start + W_VIS
    mask  = (frame_ts >= t_start) & (frame_ts < t_end)

    if mask.sum() > 0:
        pooled = frame_feat[mask].mean(axis=0)   # (512,)
        win_times.append(t_start + W_VIS / 2.0) # centre timestamp
        win_feats.append(pooled)

    t_start += S_VIS

win_times = np.array(win_times)   # (M,)
win_feats = np.stack(win_feats)   # (M, 512)

print(f'✓  Sliding windows built')
print(f'   Parameters : W = {W_VIS}s,  S = {S_VIS}s,  '
      f'overlap = {100*(W_VIS-S_VIS)/W_VIS:.0f}%')
print(f'   Windows (M): {len(win_times)}')
print(f'   Feature mat: {win_feats.shape}')


## Stage 3b — Cosine-Distance Curve and Boundary Detection

The **cosine distance** between each pair of adjacent window descriptors measures how much
the visual content changed between window $k$ and window $k+1$:

$$d_k = 1 - \cos(\mathbf{w}_k,\; \mathbf{w}_{k+1})
       = 1 - \frac{\mathbf{w}_k \cdot \mathbf{w}_{k+1}}
                  {\|\mathbf{w}_k\|\;\|\mathbf{w}_{k+1}\|}$$

A **large** $d_k$ means the visual content is strongly dissimilar across that step —
indicating a slide transition, a scene change, or the start of a new topic on screen.

Because the windows overlap heavily (and each window summarises $W_{\text{vis}}$ seconds of
frames), the curve $\{d_k\}$ changes slowly and a genuine visual transition produces a
**broad peak** rather than a single-sample spike.  We:

1. Apply Gaussian smoothing ($\sigma =$ `SMOOTH_SIGMA` steps) to further suppress noise.
2. Run `scipy.signal.find_peaks` with prominence and minimum-distance constraints to extract
   the set of **visual breakpoints** $\mathcal{B} = \{b_1, b_2, \ldots\}$ (seconds).

In [ ]:
# ── 1. Compute cosine distance between adjacent windows ─────────────────────
cos_dist = np.array([
    1.0 - cosine_sim(win_feats[i], win_feats[i + 1])
    for i in range(len(win_feats) - 1)
], dtype=np.float64)

# Timestamps of the transition midpoints
trans_times = (win_times[:-1] + win_times[1:]) / 2.0

# ── 2. Gaussian smoothing ────────────────────────────────────────────────────
cos_dist_smooth = gaussian_filter1d(cos_dist, sigma=SMOOTH_SIGMA)

# ── 3. Peak detection ────────────────────────────────────────────────────────
min_dist_steps = max(1, int(PEAK_MIN_DIST_S / S_VIS))

peak_idx, peak_props = find_peaks(
    cos_dist_smooth,
    prominence=PEAK_PROMINENCE,
    distance=min_dist_steps,
)

breakpoints = trans_times[peak_idx]   # seconds — the actual chapter boundaries

print(f'✓  Cosine-distance computation')
print(f'   Distance range (raw)    : [{cos_dist.min():.4f}, {cos_dist.max():.4f}]')
print(f'   Distance range (smooth) : [{cos_dist_smooth.min():.4f}, {cos_dist_smooth.max():.4f}]')
print(f'   Detected breakpoints    : {len(breakpoints)}')
print(f'   Breakpoint times        : {[fmt_time(b) for b in breakpoints]}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1]})

# ── Top: cosine-distance curve ───────────────────────────────────────────────
ax = axes[0]
ax.plot(trans_times, cos_dist,        color='#AAAAAA', lw=0.8, alpha=0.6, label='Raw distance')
ax.fill_between(trans_times, cos_dist_smooth, alpha=0.25, color='steelblue')
ax.plot(trans_times, cos_dist_smooth, color='steelblue', lw=1.8, label='Smoothed (σ=%.1f)' % SMOOTH_SIGMA)
ax.scatter(trans_times[peak_idx], cos_dist_smooth[peak_idx],
           color='red', zorder=5, s=60, label=f'Breakpoints (n={len(breakpoints)})')
ax.vlines(breakpoints, 0, cos_dist.max(), color='red', lw=1.0, linestyle='--', alpha=0.5)
ax.set_ylabel('Cosine distance  $d_k$', fontsize=10)
ax.set_title('Sliding-Window Cosine-Distance Curve — Direct Output of Visual Feature Analysis',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.spines[['top','right']].set_visible(False)

# ── Bottom: visual chapter blocks ────────────────────────────────────────────
ax = axes[1]
all_bounds = np.concatenate([[0.0], breakpoints, [video_end_sec]])
colours = plt.cm.tab20.colors
for i in range(len(all_bounds) - 1):
    s, e = all_bounds[i], all_bounds[i + 1]
    ax.barh(0, e - s, left=s, height=0.5,
            color=colours[i % len(colours)], edgecolor='white', linewidth=0.5)
    cx = (s + e) / 2
    if e - s > 90:
        ax.text(cx, 0, str(i + 1), ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
ax.set_yticks([])
ax.set_xlabel('Time (seconds)', fontsize=10)
ax.set_title(f'Visually-Derived Chapter Segments ({len(breakpoints)+1} chapters)', fontsize=10)
ax.spines[['top','right','left']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cosine_distance_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', os.path.join(OUTPUT_DIR, 'cosine_distance_curve.png'))


### Tuning Note

| Parameter | Effect of increasing | Effect of decreasing |
|-----------|---------------------|---------------------|
| `W_VIS` | Smoother curve, misses fast slide changes | Noisier curve, more false peaks |
| `S_VIS` | Coarser temporal resolution | Finer temporal resolution, more windows |
| `SMOOTH_SIGMA` | Fewer, more reliable peaks | More peaks, may include noise |
| `PEAK_PROMINENCE` | Fewer, stronger boundaries only | More boundaries, weaker transitions included |
| `PEAK_MIN_DIST_S` | Never two chapters shorter than this | Allows dense chapter clusters |

Re-run the two cells above after adjusting any parameter — no full re-extraction needed.

## Stage 4 — Slice Transcript at Visual Breakpoints

With the visual breakpoints $\mathcal{B}$ established, we now cut the Whisper transcript.
Each transcript segment is assigned to the chapter whose time window contains its start time.
If a resulting chapter falls below `MIN_CHAPTER_WORDS` it is merged forward into the next
chapter (prevents the LLM from receiving near-empty context).

In [ ]:
def slice_transcript(segments, breakpoints, min_words=60):
    """
    Slice `segments` at `breakpoints` (seconds) to produce chapter dicts.
    Each segment belongs to the chapter whose window contains its start_time.
    Short chapters (< min_words) are merged into the following chapter.
    """
    # Build chapter time boundaries
    bounds = np.concatenate([[0.0], np.sort(breakpoints), [float('inf')]])

    raw_chapters = []
    for i in range(len(bounds) - 1):
        t_lo, t_hi = float(bounds[i]), float(bounds[i + 1])
        segs = [s for s in segments
                if float(s['start_time']) >= t_lo and float(s['start_time']) < t_hi]
        if not segs:
            continue
        text = ' '.join(s['text'] for s in segs)
        raw_chapters.append({
            'start_sec': float(segs[0]['start_time']),
            'end_sec':   float(segs[-1]['end_time']),
            'text':      text,
            'word_count': len(text.split()),
            'n_segments': len(segs),
        })

    # Merge too-short chapters forward
    merged = []
    i = 0
    while i < len(raw_chapters):
        ch = dict(raw_chapters[i])
        while (ch['word_count'] < min_words and i + 1 < len(raw_chapters)):
            i += 1
            nxt = raw_chapters[i]
            ch['text']       += ' ' + nxt['text']
            ch['word_count'] += nxt['word_count']
            ch['end_sec']     = nxt['end_sec']
            ch['n_segments'] += nxt['n_segments']
        merged.append(ch)
        i += 1

    # Assign IDs and formatted times
    final = []
    for j, ch in enumerate(merged, start=1):
        ch['ID']         = str(j)
        ch['start_time'] = fmt_time(ch['start_sec'])
        ch['end_time']   = fmt_time(ch['end_sec'])
        final.append(ch)

    return final


chapters_visual = slice_transcript(segments, breakpoints, MIN_CHAPTER_WORDS)

durations = [(c['end_sec'] - c['start_sec']) / 60 for c in chapters_visual]
words     = [c['word_count'] for c in chapters_visual]

print(f'✓  Transcript sliced into {len(chapters_visual)} visual chapters')
print(f'   Words/chapter : mean={np.mean(words):.0f}, '
      f'min={min(words)}, max={max(words)}')
print(f'   Duration/ch   : mean={np.mean(durations):.2f} min, '
      f'min={min(durations):.2f}, max={max(durations):.2f}')
print()
print(f'   First 5 chapters:')
for c in chapters_visual[:5]:
    print(f'     [{c["ID"]:>3}] {c["start_time"]} → {c["end_time"]}  '
          f'({c["word_count"]} words)')


## Stage 4b — Visual Feature Vector per Chapter

Now that we have chapters whose boundaries come from the visual signal, we assign
a visual descriptor to each chapter by mean-pooling the **sliding-window features**
whose centre timestamps fall inside the chapter.  Using window-level features
(rather than raw per-frame features) gives a smoother, context-aware visual descriptor
because every window already aggregates `W_VIS` seconds of frames.

In [ ]:
chapter_visual_feats = []
for ch in chapters_visual:
    s, e   = ch['start_sec'], ch['end_sec']
    mask   = (win_times >= s) & (win_times <= e)
    if mask.sum() > 0:
        feat = win_feats[mask].mean(axis=0)          # mean-pool window features
    else:
        nearest = int(np.argmin(np.abs(win_times - (s + e) / 2)))
        feat = win_feats[nearest]                    # fallback: nearest window
    chapter_visual_feats.append(feat)

print(f'✓  Visual descriptor assigned to each chapter')
print(f'   Shape: ({len(chapter_visual_feats)}, {chapter_visual_feats[0].shape[0]})')


## Stage 5 — SBERT Encoding of Visually-Defined Chapters

In [ ]:
print(f'Loading SBERT model: {SBERT_MODEL}')
sbert = SentenceTransformer(SBERT_MODEL)

texts = [c['text'][:1200] for c in chapters_visual]
sbert_embs = sbert.encode(
    texts, batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'\n✓  SBERT embeddings: {sbert_embs.shape}')


## Evaluation — Within-Chapter Visual Coherence

Because the chapter boundaries were derived from the visual signal, we expect strong
**within-chapter visual coherence**: all frames inside a chapter should look similar to
the chapter's mean visual feature.  We compute this as

$$C_\text{vis}(i) = \frac{1}{|\mathcal{W}_i|}\sum_{k \in \mathcal{W}_i}
   \cos(\mathbf{w}_k,\; \bar{\mathbf{w}}_i)$$

where $\mathcal{W}_i$ is the set of sliding windows inside chapter $i$ and
$\bar{\mathbf{w}}_i$ is their mean pool.

We also compare **between-chapter SBERT similarity** against the original word-count pipeline
output (loaded from `CHAPTERS_ORIG_PATH`) to assess whether visual boundaries produce more
semantically distinct chapters.

In [ ]:
# ── Within-chapter visual coherence ─────────────────────────────────────────
vis_coh = []
for ch, ch_feat in zip(chapters_visual, chapter_visual_feats):
    s, e   = ch['start_sec'], ch['end_sec']
    mask   = (win_times >= s) & (win_times <= e)
    if mask.sum() < 2:
        vis_coh.append(1.0)
        continue
    sims = [cosine_sim(win_feats[j], ch_feat) for j in np.where(mask)[0]]
    vis_coh.append(float(np.mean(sims)))

vis_coh = np.array(vis_coh)

# ── Between-chapter SBERT similarity ────────────────────────────────────────
between_sim = np.array([
    cosine_sim(sbert_embs[i], sbert_embs[i+1])
    for i in range(len(sbert_embs) - 1)
])

print('Visual within-chapter coherence')
print(f'  mean = {vis_coh.mean():.4f},  std = {vis_coh.std():.4f}')
print(f'  min  = {vis_coh.min():.4f},   max = {vis_coh.max():.4f}')
print()
print('Between-chapter SBERT similarity (lower = more distinct chapters)')
print(f'  mean = {between_sim.mean():.4f},  std = {between_sim.std():.4f}')

# ── Load original pipeline output for comparison (if available) ──────────────
orig_path = 'Output/shortened-chapters.json'
if os.path.exists(orig_path):
    with open(orig_path) as f:
        orig_chs = json.load(f)
    orig_texts = [(c.get('chapter','') + ' ' + c.get('description','')).strip()
                  for c in orig_chs]
    orig_texts = [t if t.strip() else 'placeholder' for t in orig_texts]
    orig_embs  = sbert.encode(orig_texts, batch_size=16,
                               show_progress_bar=False, convert_to_numpy=True)
    orig_between = np.array([
        cosine_sim(orig_embs[i], orig_embs[i+1])
        for i in range(len(orig_embs)-1)
    ])
    print(f'\nOriginal pipeline between-chapter SBERT similarity')
    print(f'  mean = {orig_between.mean():.4f}')
    print(f'  → Δ (visual minus original) = {between_sim.mean()-orig_between.mean():+.4f}')
    print(f'    (negative = visual chapters are MORE distinct)')
else:
    orig_between = None
    print('Original chapters.json not found — skipping comparison')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# ── (a) Visual coherence per chapter ────────────────────────────────────────
ax = axes[0]
bar_cols = ['#2E75B6' if v >= 0.85 else '#ED7D31' for v in vis_coh]
ax.bar(range(len(vis_coh)), vis_coh, color=bar_cols, edgecolor='white')
ax.axhline(vis_coh.mean(), color='red', lw=1.5, ls='--',
           label=f'Mean = {vis_coh.mean():.4f}')
ax.set_xlabel('Chapter index'); ax.set_ylabel('Visual coherence')
ax.set_title('Within-Chapter Visual Coherence\n(visual-first pipeline)')
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)
ax.spines[['top','right']].set_visible(False)

# ── (b) Between-chapter SBERT comparison ────────────────────────────────────
ax = axes[1]
ax.plot(between_sim, 's-', color='#538135', lw=1.5, ms=4,
        label=f'Visual-first  (μ={between_sim.mean():.3f})')
if orig_between is not None:
    n = min(len(orig_between), len(between_sim))
    ax.plot(orig_between[:n], 'o--', color='steelblue', lw=1.2, ms=4, alpha=0.7,
            label=f'Word-count    (μ={orig_between.mean():.3f})')
ax.set_xlabel('Chapter boundary index'); ax.set_ylabel('Cosine similarity')
ax.set_title('Between-Chapter SBERT Similarity\n(lower = more distinct)')
ax.legend(fontsize=8); ax.set_ylim(0, 1)
ax.spines[['top','right']].set_visible(False)

# ── (c) Chapter duration distribution ───────────────────────────────────────
ax = axes[2]
durations_min = [(c['end_sec'] - c['start_sec']) / 60 for c in chapters_visual]
ax.hist(durations_min, bins=12, color='#4472C4', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(durations_min), color='red', lw=1.5, ls='--',
           label=f'Mean = {np.mean(durations_min):.2f} min')
ax.set_xlabel('Chapter duration (minutes)')
ax.set_ylabel('Count')
ax.set_title('Chapter Duration Distribution\n(visual segmentation)')
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'evaluation_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()


## Stage 6 — LLM Chapter Label Generation (Qwen2.5-1.5B-Instruct)

In [ ]:
print(f'Loading {LLM_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float32,         # torch_dtype deprecated in transformers 5.x
    trust_remote_code=True,
)
llm.eval()
print('LLM ready')


In [ ]:
def label_chapter(ch):
    """
    Zero-shot prompt → structured JSON chapter annotation.
    Identical prompt format to main pipeline for direct comparability.
    """
    dur    = ch['end_sec'] - ch['start_sec']
    prompt = (
        f"Generate a chapter summary JSON for this lecture segment.\n\n"
        f"Time Range : {ch['start_time']} to {ch['end_time']}\n"
        f"Duration   : {dur:.1f} seconds\n"
        f"Word Count : {ch['word_count']}\n\n"
        f"Transcript :\n{ch['text'][:600]}\n\n"
        f"Requirements:\n"
        f"- chapter     : 5-8 word descriptive title\n"
        f"- description : 1-2 sentence technical summary\n\n"
        f"Output ONLY valid JSON. No markdown. No explanation."
    )
    messages = [
        {'role': 'system', 'content': 'You are a precise JSON-generating assistant.'},
        {'role': 'user',   'content': prompt},
    ]
    raw_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(raw_input, return_tensors='pt')

    for attempt in range(MAX_RETRIES):
        with torch.no_grad():
            out = llm.generate(
                **inputs,
                max_new_tokens=LLM_MAX_TOKENS,
                temperature=LLM_TEMPERATURE,
                top_p=LLM_TOP_P,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        json_str = extract_json_brace(generated)
        if json_str:
            try:
                obj = json.loads(json_str)
                if 'chapter' in obj and 'description' in obj:
                    return obj
            except json.JSONDecodeError:
                pass
        print(f'    Retry {attempt+1}/{MAX_RETRIES} — ch {ch["ID"]}')

    # Fallback: use first 7 words as title
    return {
        'chapter':     ' '.join(ch['text'].split()[:7]),
        'description': 'Fallback: LLM parse failed after max retries.'
    }


print('Label function defined')


In [ ]:
print(f'Generating labels for {len(chapters_visual)} chapters...')
t0 = time.time()

labelled_chapters = []
for ch in tqdm(chapters_visual, desc='LLM labelling'):
    label = label_chapter(ch)
    lv    = is_low_value(label.get('chapter', ''), label.get('description', ''))
    labelled_chapters.append({
        'ID':              ch['ID'],
        'chapter':         label.get('chapter', ''),
        'description':     label.get('description', ''),
        'start_time':      ch['start_time'],
        'end_time':        ch['end_time'],
        'start_sec':       ch['start_sec'],
        'end_sec':         ch['end_sec'],
        'word_count':      ch['word_count'],
        'key':             str(not lv),
        'boundary_source': 'visual_cosine_sliding_window',
        'vis_coherence':   round(float(vis_coh[int(ch['ID'])-1]), 4),
    })

elapsed = time.time() - t0
print(f'\n✓  LLM labelling done: {elapsed/60:.1f} min  '
      f'({elapsed/len(chapters_visual):.0f}s / chapter)')


## Save All Outputs

In [ ]:
# -- Save chapters in BOTH notebook-analysis and Flask-compatible formats --

# Derive video stem from VIDEO_PATH for Flask-compatible naming
from pathlib import Path as _Path
_video_stem = _Path(VIDEO_PATH).stem

# -- chapters_sw_visual.json (notebook analysis) --
out_ch = os.path.join(OUTPUT_DIR, 'chapters_sw_visual.json')
with open(out_ch, 'w', encoding='utf-8') as f:
    json.dump(labelled_chapters, f, indent=2, ensure_ascii=False)

# -- {stem}-chapters.json (Flask app compatible) --
flask_out = os.path.join('Output', f'{_video_stem}-chapters.json')
with open(flask_out, 'w', encoding='utf-8') as f:
    json.dump(labelled_chapters, f, indent=2, ensure_ascii=False)
print(f'Flask-compatible output: {flask_out}')

# -- pipeline_summary.json --
summary = {
    'pipeline': 'Visual-First Sliding Window Segmentation',
    'parameters': {
        'FRAME_SAMPLE_RATE_s': FRAME_SAMPLE_RATE,
        'W_VIS_s':             W_VIS,
        'S_VIS_s':             S_VIS,
        'overlap_pct':         round(100*(W_VIS-S_VIS)/W_VIS, 1),
        'SMOOTH_SIGMA':        SMOOTH_SIGMA,
        'PEAK_PROMINENCE':     PEAK_PROMINENCE,
        'PEAK_MIN_DIST_S':     PEAK_MIN_DIST_S,
        'MIN_CHAPTER_WORDS':   MIN_CHAPTER_WORDS,
    },
    'results': {
        'n_visual_breakpoints':     int(len(breakpoints)),
        'n_chapters_after_merge':   int(len(labelled_chapters)),
        'mean_words_per_chapter':   float(np.mean([c['word_count'] for c in labelled_chapters])),
        'mean_duration_min':        float(np.mean([(c['end_sec']-c['start_sec'])/60
                                                    for c in labelled_chapters])),
        'visual_coherence_mean':    float(vis_coh.mean()),
        'visual_coherence_std':     float(vis_coh.std()),
        'between_ch_sbert_mean':    float(between_sim.mean()),
        'orig_between_ch_mean':     float(orig_between.mean()) if orig_between is not None else None,
    }
}
out_sum = os.path.join(OUTPUT_DIR, 'pipeline_summary.json')
with open(out_sum, 'w') as f:
    json.dump(summary, f, indent=2)

print('Files saved:')
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp):
        print(f'   {fn:<45}  {os.path.getsize(fp)//1024:>4} KB')

print()
print('Summary:')
for k, v in summary['results'].items():
    print(f'  {k:<35} {v}')


## Flask Web Application

The output JSON is **schema-compatible** with the main pipeline. Two ways to serve it:

### Option A: Visual-first Flask app (recommended)

```bash
make run-visual      # launches sw_visual_app.py at http://localhost:5000
```

This runs a Flask app that uses the **sliding-window visual pipeline** as its processing
backend. Upload a video or click "Generate Chapters" on an existing video to run the
full visual segmentation pipeline from the browser.

### Option B: Use with the original text-first Flask app

This notebook now saves its output to **both**:
- `Output/sw_visual/chapters_sw_visual.json` (notebook analysis)
- `Output/{stem}-chapters.json` (Flask-compatible)

After running this notebook, start the original Flask app:

```bash
make run             # launches app.ipynb at http://localhost:5000
```

The visual chapters will appear automatically because they are saved in the standard
output path.

---

## Pipeline Comparison Summary

| | Main pipeline (`make run`) | Visual pipeline (`make run-visual`) |
|---|---|---|
| **Boundary source** | Word count (300w tumbling) | Cosine distance peaks on visual features |
| **Segmentation unit** | Text (words) | Time (seconds via visual change) |
| **Visual features role** | Descriptive (aligned to text chapters) | **Generative (defines boundaries)** |
| **Sliding window** | None | W=60s / S=5s on ResNet-18 vectors |
| **Cosine detection** | None | Applied to adjacent window descriptors |
| **Chapter count** | Fixed (~N/300) | Data-driven (peaks in distance curve) |
| **Short chapter handling** | N/A | Merge-forward if < MIN_CHAPTER_WORDS |
| **Within-ch visual coherence** | Not guaranteed | Maximised by design |
| **Output JSON** | shortened-chapters.json | {stem}-chapters.json (same schema) |